In [1]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [3]:
#Load Dataset

df = pd.read_excel("/Users/karthikprasannakumar/Downloads/Project_2_Neobank_Churn_Engagement_Data.xlsx")

print(df.head())
print(df.shape)

   Customer_ID Signup_Date Snapshot_Month         Country  Age Income_Band  \
0  CUST-200000  2025-08-11     2026-05-01        Pakistan   70   Lower-Mid   
1  CUST-200001  2022-06-20     2026-05-01          Canada   41   Lower-Mid   
2  CUST-200002  2023-11-29     2026-05-01          Canada   51         Low   
3  CUST-200003  2022-10-15     2026-05-01  United Kingdom   45   Lower-Mid   
4  CUST-200004  2024-10-08     2026-05-01  United Kingdom   62         Low   

  Occupation_Type Plan_Type KYC_Status  Tenure_Months  ...  Virtual_Card_Used  \
0         Retired  Business  Completed              8  ...                  1   
1      Freelancer      Free  Completed             47  ...                  0   
2         Retired   Premium  Completed             29  ...                  0   
3        Salaried      Free    Pending             43  ...                  0   
4         Retired      Free    Pending             19  ...                  1   

   Credit_Score_Band  NPS  Churned_90D  Core

In [5]:
# Convert Signup Date

df["Signup_Date"] = pd.to_datetime(
    df["Signup_Date"],
    dayfirst=True,
    errors="coerce"
)

print(df["Signup_Date"].head())

0   2025-08-11
1   2022-06-20
2   2023-11-29
3   2022-10-15
4   2024-10-08
Name: Signup_Date, dtype: datetime64[ns]


In [9]:
# Convert Numeric Columns

numeric_columns = [

    "Account_Balance",
    "Monthly_Deposits",
    "Monthly_Transactions",
    "App_Sessions_30D",
    "Core_Feature_Score",
    "Failed_Logins",
    "Support_Tickets",
    "Engagement_Rate"

]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print(df.dtypes)

Customer_ID                           object
Signup_Date                   datetime64[ns]
Snapshot_Month                datetime64[ns]
Country                               object
Age                                    int64
Income_Band                           object
Occupation_Type                       object
Plan_Type                             object
KYC_Status                            object
Tenure_Months                          int64
Account_Balance                      float64
Monthly_Deposits                     float64
Monthly_Transactions                   int64
Card_Txn_Count                         int64
P2P_Transfers                          int64
Bill_Payments                          int64
App_Sessions_30D                       int64
Failed_Logins                          int64
Support_Tickets                        int64
Cashback_Used                          int64
Savings_Pots_Used                      int64
Virtual_Card_Used                      int64
Credit_Sco

In [11]:
#Intelligent Filter

quarantine_mask = (

    (df["Monthly_Transactions"] > 0)

    &

    (df["App_Sessions_30D"] == 0)

)

quarantine_df = df[quarantine_mask].copy()

clean_df = df[~quarantine_mask].copy()

print("Corrupted Records :", len(quarantine_df))
print("Clean Records :", len(clean_df))

Corrupted Records : 45
Clean Records : 2005


In [13]:
# Export Corrupted Records

quarantine_df.to_csv(

    "quarantined_churn_data.csv",

    index=False

)

print("Quarantine dataset exported successfully.")

Quarantine dataset exported successfully.


In [15]:
# AI Risk Band Recalculation

conditions = [

    clean_df["Core_Feature_Score"] < 40,

    clean_df["Core_Feature_Score"].between(40,69),

    clean_df["Core_Feature_Score"] >= 70

]

choices = [

    "High",

    "Medium",

    "Low"

]

clean_df["AI_Risk_Band"] = np.select(

    conditions,

    choices,

    default="Medium"

)

clean_df[["Core_Feature_Score","AI_Risk_Band"]].head()

,Core_Feature_Score,AI_Risk_Band
0,96,Low
1,39,High
2,83,Low
3,45,Medium
4,70,Low


In [17]:
# Customer Friction Score


clean_df["Customer_Friction_Score"] = (

    100

    +

    np.where(

        clean_df["Plan_Type"]=="Premium",

        10,

        0

    )

    -

    (clean_df["Failed_Logins"]*25)

    -

    np.where(

        clean_df["KYC_Status"]=="Pending",

        40,

        0

    )

    -

    (clean_df["Support_Tickets"]*5)

)

clean_df["Customer_Friction_Score"] = (

    clean_df["Customer_Friction_Score"]

    .clip(lower=0)

)

clean_df[

    [

        "Customer_Friction_Score"

    ]

].head()

,Customer_Friction_Score
0,75
1,25
2,60
3,60
4,35


In [19]:

# Transaction Volatility

volatility = (

    clean_df

    .groupby("Plan_Type")["Monthly_Transactions"]

    .agg(

        Mean="mean",

        Std="std"

    )

)

volatility["Plan_Transaction_Volatility_Index"] = (

    volatility["Std"]

    /

    volatility["Mean"]

)

clean_df = clean_df.merge(

    volatility[
        ["Plan_Transaction_Volatility_Index"]
    ],

    on="Plan_Type",

    how="left"

)

clean_df.head()

,Customer_ID,Signup_Date,Snapshot_Month,Country,Age,Income_Band,Occupation_Type,Plan_Type,KYC_Status,Tenure_Months,...,NPS,Churned_90D,Core_Feature_Score,Engagement_Rate,AI_Churn_Risk_Score,AI_Risk_Band,Predicted_Next_Best_Action,Core_Feature_Gap,Customer_Friction_Score,Plan_Transaction_Volatility_Index
0,CUST-200000,2025-08-11,2026-05-01,Pakistan,70,Lower-Mid,Retired,Business,Completed,8,...,59,0,96,1.578947,0,Low,Upsell premium bundle,4,75,0.366649
1,CUST-200001,2022-06-20,2026-05-01,Canada,41,Lower-Mid,Freelancer,Free,Completed,47,...,13,1,39,0.402597,31,High,Upsell premium bundle,61,25,0.463090
2,CUST-200002,2023-11-29,2026-05-01,Canada,51,Low,Retired,Premium,Completed,29,...,-9,0,83,0.949153,16,Low,Upsell premium bundle,17,60,0.281762
3,CUST-200003,2022-10-15,2026-05-01,United Kingdom,45,Lower-Mid,Salaried,Free,Pending,43,...,59,0,45,0.397260,27,Medium,Upsell premium bundle,55,60,0.463090
4,CUST-200004,2024-10-08,2026-05-01,United Kingdom,62,Low,Retired,Free,Pending,19,...,25,0,70,0.979592,32,Low,Upsell premium bundle,30,35,0.463090


In [21]:
#Connect MySQL

username = "root"

password = quote_plus("K@rthik18")

host = "127.0.0.1"

port = 3306

database = "marketing_db"

engine = create_engine(

    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"

)

with engine.connect() as conn:

    print("Connected Successfully!")

Connected Successfully!


In [23]:
# Step 5 - Load into Database


clean_df.to_sql(

    name="neobank_customer_churn",

    con=engine,

    if_exists="replace",

    index=False

)

print("Dataset loaded successfully into MySQL.")

Dataset loaded successfully into MySQL.


In [25]:
# Final Dataset


print(clean_df.shape)

clean_df.head()

(2005, 33)


,Customer_ID,Signup_Date,Snapshot_Month,Country,Age,Income_Band,Occupation_Type,Plan_Type,KYC_Status,Tenure_Months,...,NPS,Churned_90D,Core_Feature_Score,Engagement_Rate,AI_Churn_Risk_Score,AI_Risk_Band,Predicted_Next_Best_Action,Core_Feature_Gap,Customer_Friction_Score,Plan_Transaction_Volatility_Index
0,CUST-200000,2025-08-11,2026-05-01,Pakistan,70,Lower-Mid,Retired,Business,Completed,8,...,59,0,96,1.578947,0,Low,Upsell premium bundle,4,75,0.366649
1,CUST-200001,2022-06-20,2026-05-01,Canada,41,Lower-Mid,Freelancer,Free,Completed,47,...,13,1,39,0.402597,31,High,Upsell premium bundle,61,25,0.463090
2,CUST-200002,2023-11-29,2026-05-01,Canada,51,Low,Retired,Premium,Completed,29,...,-9,0,83,0.949153,16,Low,Upsell premium bundle,17,60,0.281762
3,CUST-200003,2022-10-15,2026-05-01,United Kingdom,45,Lower-Mid,Salaried,Free,Pending,43,...,59,0,45,0.397260,27,Medium,Upsell premium bundle,55,60,0.463090
4,CUST-200004,2024-10-08,2026-05-01,United Kingdom,62,Low,Retired,Free,Pending,19,...,25,0,70,0.979592,32,Low,Upsell premium bundle,30,35,0.463090
